# 05 — Inventory policy and sensitivity audit

The backend pipeline is the canonical source for policy optimization and simulation. Run `python src/backend/main.py` from the repository root before using this notebook. Every cell below is read-only: it inspects pipeline artifacts and does not rebuild policies or persist data.

Decision contract:

- The primary historical baseline uses the **recorded reorder point and order quantity**. Receipt-inferred ROP is shown only as a sensitivity case.
- The proposed in-scope policy jointly searches residual-quantile ROP and order quantity `Q`, subject to MOQ and order-multiple constraints.
- During daily review at the end of day `t`, proposed dynamic ROP uses only the forecast whose target date is `t+1`; the terminal day places no new order.
- Old and proposed policies share the same locked-evaluation demand path and initial on-hand inventory.
- All savings are modeled counterfactual scenario results under documented cost assumptions, not realized financial savings.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "backend").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "backend").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

ARTIFACT_PATHS = {
    "policy_sku": ROOT / "data" / "processed" / "policy_sku.csv",
    "old_policy_metric": ROOT / "data" / "processed" / "old_policy_metric.csv",
    "new_policy_metric": ROOT / "data" / "processed" / "new_policy_metric.csv",
    "policy_candidate_audit": ROOT / "data" / "processed" / "policy_candidate_audit.csv",
    "policy_action": ROOT / "data" / "processed" / "policy_action.csv",
    "policy_sensitivity": ROOT / "data" / "processed" / "policy_sensitivity.csv",
    "historical_policy_sensitivity": ROOT / "data" / "processed" / "historical_policy_sensitivity.csv",
    "full_policy_summary": ROOT / "data" / "processed" / "full_policy_summary.csv",
    "scenario_uncertainty": ROOT / "data" / "processed" / "scenario_uncertainty.csv",
}
missing = [str(path.relative_to(ROOT)) for path in ARTIFACT_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing canonical policy artifacts. Run `python src/backend/main.py`: "
        + ", ".join(missing)
    )

artifacts = {name: pd.read_csv(path) for name, path in ARTIFACT_PATHS.items()}
policy_sku = artifacts["policy_sku"]
old_policy_metric = artifacts["old_policy_metric"]
new_policy_metric = artifacts["new_policy_metric"]
policy_candidate_audit = artifacts["policy_candidate_audit"]
policy_action = artifacts["policy_action"]
policy_sensitivity = artifacts["policy_sensitivity"]
historical_policy_sensitivity = artifacts["historical_policy_sensitivity"]
full_policy_summary = artifacts["full_policy_summary"]
scenario_uncertainty = artifacts["scenario_uncertainty"]

pd.DataFrame(
    [{"artifact": name, "rows": len(table), "columns": len(table.columns)} for name, table in artifacts.items()]
)

## Policy lineage checks

These checks verify labels and fields already written by the backend. They do not infer a replacement policy inside the notebook.

In [ ]:
lineage_checks = pd.DataFrame(
    [
        {
            "contract": "Primary baseline is recorded ROP",
            "passed": bool(
                "baseline_source" in historical_policy_sensitivity.columns
                and historical_policy_sensitivity["baseline_source"].eq("recorded_reorder_point").any()
            ),
        },
        {
            "contract": "Receipt-inferred ROP is sensitivity only",
            "passed": bool(
                "baseline_source" in historical_policy_sensitivity.columns
                and historical_policy_sensitivity["baseline_source"].eq("receipt_inferred_proxy").any()
            ),
        },
        {
            "contract": "Proposed policy uses dynamic t+1 ROP",
            "passed": bool(
                "policy_mode" in policy_action.columns
                and policy_action["policy_mode"].eq("daily_dynamic_rop_next_day_forecast").all()
            ),
        },
        {
            "contract": "Order quantity Q is grid-selected",
            "passed": {"selected_q_multiplier", "order_quantity"}.issubset(policy_sku.columns),
        },
        {
            "contract": "Candidate audit records MOQ and order multiple",
            "passed": {"minimum_order_quantity", "order_multiple"}.issubset(policy_candidate_audit.columns),
        },
    ]
)
lineage_checks

## Canonical policy results

`full_policy_summary.csv` is the portfolio view across all SKUs: out-of-scope SKUs retain their recorded policy. The in-scope old/new metric artifacts support a narrower intervention comparison.

In [ ]:
metric_columns = [
    "policy",
    "sku_id",
    "class",
    "total_demand",
    "total_sales",
    "fill_rate",
    "cycle_service_level",
    "avg_inventory",
    "stockout_cost",
    "holding_cost",
    "ordering_cost",
    "total_cost",
]
in_scope_metric_rows = pd.concat(
    [
        old_policy_metric.assign(policy="recorded_policy_in_scope"),
        new_policy_metric.assign(policy="proposed_dynamic_policy_in_scope"),
    ],
    ignore_index=True,
)
display(full_policy_summary)
display(in_scope_metric_rows[metric_columns].sort_values(["sku_id", "policy"]).head(40))
display(historical_policy_sensitivity)

In [ ]:
candidate_summary = (
    policy_candidate_audit.groupby("sku_id", observed=True)
    .agg(
        candidate_count=("q_multiplier", "size"),
        feasible_count=("service_floor_met", "sum"),
        min_modeled_cost=("total_cost", "min"),
        min_order_quantity=("order_quantity", "min"),
        max_order_quantity=("order_quantity", "max"),
    )
    .reset_index()
)
selected_columns = [
    column
    for column in [
        "sku_id",
        "class",
        "selected_q_multiplier",
        "selected_safety_stock_quantile",
        "order_quantity",
        "safety_stock",
        "rop",
        "calibration_fill_rate",
        "calibration_service_floor",
        "selection_status",
    ]
    if column in policy_sku.columns
]
action_columns = [
    column
    for column in [
        "priority_rank",
        "sku_id",
        "class",
        "intervention",
        "recommended_action",
        "current_rop",
        "proposed_rop",
        "current_order_quantity",
        "proposed_order_quantity",
        "fill_delta_pp",
        "avg_on_hand_value_delta",
        "modeled_cost_saving",
        "selection_status",
    ]
    if column in policy_action.columns
]
display(candidate_summary.describe(include="all"))
display(policy_sku[selected_columns].head(20))
display(policy_action.sort_values("priority_rank")[action_columns].head(20))

## Assumption and scenario sensitivity

`policy_sensitivity.csv` reprices the same paired paths over ordering, holding, and shortage-cost assumptions. `scenario_uncertainty.csv` perturbs forecast-anchored demand and lead time; its percentiles describe scenario sensitivity, not prediction intervals or causal confidence intervals.

In [ ]:
cost_metrics = [
    column
    for column in [
        "fill_rate_delta_percentage_points",
        "avg_inventory_change_pct",
        "total_cost_savings",
        "total_cost_change_pct",
    ]
    if column in policy_sensitivity.columns
]
cost_sensitivity_summary = policy_sensitivity[cost_metrics].agg(["min", "median", "max"]).T

scenario_metrics = [
    column
    for column in [
        "fill_rate_delta_percentage_points",
        "avg_inventory_change_pct",
        "total_cost_savings",
        "total_cost_change_pct",
    ]
    if column in scenario_uncertainty.columns
]
scenario_percentiles = (
    scenario_uncertainty[scenario_metrics]
    .quantile([0.05, 0.50, 0.95])
    .rename(index={0.05: "p05", 0.50: "p50", 0.95: "p95"})
    .T
)
display(cost_sensitivity_summary)
display(scenario_percentiles)
display(policy_sensitivity.head(12))
display(scenario_uncertainty.head(12))